<div style="background:#E9FFF6; color:#440404; padding:8px; border-radius: 4px; text-align: center; font-weight: 500;">IFN619 - Data Analytics for Strategic Decision Makers</div>

# IFN619 :: B3-Semi/unstructured Analytics (Part A)

For this session, the focus will be on analysis of unstructured text. However, the thinking required is similar to approaches to analysing images, video, sound and other unstructured data. Primarily, the analysis is based on the notion that there are useful patterns in the unstructured data which can be obtained computationally.

Most of the time, working with semi-structured or unstructured data involves a process of incrementally structuring it to the point where we can obtain meaningful information. Algorithms like topic modelling algorithms (which we will look at in Part B) allow more complex analysis of semi/unstructured data. For now, we will look at some basic approaches.

In [1]:
# Import the necessary libraries
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import pandas as pd
import json
import random

### Basic incremental structuring

Before using the imported Python libraries, let's look at how we can work with unstructured text data by taking a simple (unstructured) string of characters and modify it to be more structured.

In [2]:
# selected from David Bohm's 1990 paper: "A new theory of the relationship of mind and matter" (p.281)

bohm_text = "Thus, for example, when we read a printed page, we do not assimilate the substance of the paper, but only the forms of the letters, and it is these forms which give rise to an information content in the reader which is manifested actively in his or her subsequent activities. A similar mind-like quality of matter reveals itself strongly at the quantum level, in the sense that the form of the wave function manifests itself in the movements of the particles. This quality does not, however, appear to a significant extent at the level at which classical physics is a valid approximation."

# show the contents of the variable
bohm_text

'Thus, for example, when we read a printed page, we do not assimilate the substance of the paper, but only the forms of the letters, and it is these forms which give rise to an information content in the reader which is manifested actively in his or her subsequent activities. A similar mind-like quality of matter reveals itself strongly at the quantum level, in the sense that the form of the wave function manifests itself in the movements of the particles. This quality does not, however, appear to a significant extent at the level at which classical physics is a valid approximation.'

We can split the string into separate sentences using the `split()` function.

In [3]:
# Use an appropriate character to split text into sentences
bohm_sents = bohm_text.split('.')
bohm_sents

['Thus, for example, when we read a printed page, we do not assimilate the substance of the paper, but only the forms of the letters, and it is these forms which give rise to an information content in the reader which is manifested actively in his or her subsequent activities',
 ' A similar mind-like quality of matter reveals itself strongly at the quantum level, in the sense that the form of the wave function manifests itself in the movements of the particles',
 ' This quality does not, however, appear to a significant extent at the level at which classical physics is a valid approximation',
 '']

We can then process each sentence to extract words, adding them to a list

In [4]:
bohm_words = []
for sent in bohm_sents:
    # Use an appropriate character to split sentences into words
    sent_words = sent.split(' ')
    for word in sent_words:
        bohm_words.append(word)
        
bohm_words

['Thus,',
 'for',
 'example,',
 'when',
 'we',
 'read',
 'a',
 'printed',
 'page,',
 'we',
 'do',
 'not',
 'assimilate',
 'the',
 'substance',
 'of',
 'the',
 'paper,',
 'but',
 'only',
 'the',
 'forms',
 'of',
 'the',
 'letters,',
 'and',
 'it',
 'is',
 'these',
 'forms',
 'which',
 'give',
 'rise',
 'to',
 'an',
 'information',
 'content',
 'in',
 'the',
 'reader',
 'which',
 'is',
 'manifested',
 'actively',
 'in',
 'his',
 'or',
 'her',
 'subsequent',
 'activities',
 '',
 'A',
 'similar',
 'mind-like',
 'quality',
 'of',
 'matter',
 'reveals',
 'itself',
 'strongly',
 'at',
 'the',
 'quantum',
 'level,',
 'in',
 'the',
 'sense',
 'that',
 'the',
 'form',
 'of',
 'the',
 'wave',
 'function',
 'manifests',
 'itself',
 'in',
 'the',
 'movements',
 'of',
 'the',
 'particles',
 '',
 'This',
 'quality',
 'does',
 'not,',
 'however,',
 'appear',
 'to',
 'a',
 'significant',
 'extent',
 'at',
 'the',
 'level',
 'at',
 'which',
 'classical',
 'physics',
 'is',
 'a',
 'valid',
 'approximatio

We can clean up this list by removing empty elements, dropping commas, and making all words lowercase

In [5]:
# Only keep words that are not '' (empty string), make them lower case and remove any comma characters
bohm_clean_words = [w.lower().replace(',','') for w in bohm_words if w!='']
bohm_clean_words

['thus',
 'for',
 'example',
 'when',
 'we',
 'read',
 'a',
 'printed',
 'page',
 'we',
 'do',
 'not',
 'assimilate',
 'the',
 'substance',
 'of',
 'the',
 'paper',
 'but',
 'only',
 'the',
 'forms',
 'of',
 'the',
 'letters',
 'and',
 'it',
 'is',
 'these',
 'forms',
 'which',
 'give',
 'rise',
 'to',
 'an',
 'information',
 'content',
 'in',
 'the',
 'reader',
 'which',
 'is',
 'manifested',
 'actively',
 'in',
 'his',
 'or',
 'her',
 'subsequent',
 'activities',
 'a',
 'similar',
 'mind-like',
 'quality',
 'of',
 'matter',
 'reveals',
 'itself',
 'strongly',
 'at',
 'the',
 'quantum',
 'level',
 'in',
 'the',
 'sense',
 'that',
 'the',
 'form',
 'of',
 'the',
 'wave',
 'function',
 'manifests',
 'itself',
 'in',
 'the',
 'movements',
 'of',
 'the',
 'particles',
 'this',
 'quality',
 'does',
 'not',
 'however',
 'appear',
 'to',
 'a',
 'significant',
 'extent',
 'at',
 'the',
 'level',
 'at',
 'which',
 'classical',
 'physics',
 'is',
 'a',
 'valid',
 'approximation']

Finally, we can count how many of each word we have to get an indication of David Bohm's vocabulary used in this quote

In [6]:
bohm_vocab = {}
for word in bohm_clean_words:
    bohm_vocab[word] = bohm_vocab.get(word, 0) + 1

# Sort vocab by count (value)
bohm_vocab_sorted = dict(sorted(bohm_vocab.items(), key=lambda item: item[1]))

# Show the final dictionary
bohm_vocab_sorted

{'thus': 1,
 'for': 1,
 'example': 1,
 'when': 1,
 'read': 1,
 'printed': 1,
 'page': 1,
 'do': 1,
 'assimilate': 1,
 'substance': 1,
 'paper': 1,
 'but': 1,
 'only': 1,
 'letters': 1,
 'and': 1,
 'it': 1,
 'these': 1,
 'give': 1,
 'rise': 1,
 'an': 1,
 'information': 1,
 'content': 1,
 'reader': 1,
 'manifested': 1,
 'actively': 1,
 'his': 1,
 'or': 1,
 'her': 1,
 'subsequent': 1,
 'activities': 1,
 'similar': 1,
 'mind-like': 1,
 'matter': 1,
 'reveals': 1,
 'strongly': 1,
 'quantum': 1,
 'sense': 1,
 'that': 1,
 'form': 1,
 'wave': 1,
 'function': 1,
 'manifests': 1,
 'movements': 1,
 'particles': 1,
 'this': 1,
 'does': 1,
 'however': 1,
 'appear': 1,
 'significant': 1,
 'extent': 1,
 'classical': 1,
 'physics': 1,
 'valid': 1,
 'approximation': 1,
 'we': 2,
 'not': 2,
 'forms': 2,
 'to': 2,
 'quality': 2,
 'itself': 2,
 'level': 2,
 'is': 3,
 'which': 3,
 'at': 3,
 'a': 4,
 'in': 4,
 'of': 5,
 'the': 12}

What do you notice about the words in the vocabulary? 
Which words are meaningful? Which are not?

How do you think we could extract the meaning from the unstructured text?

---

## Using vectorizors to work with unstructured text

#### Accessing the data via The Guardian API

See the `Accessing_the_Guardian_API.ipynb` notebook file for details on getting the data. **Note:** This approach may be used for additional data for Assignment 2.

### Read in pre-saved data

To save time, we're loading in pre-saved data that was fetched using the Guardian API. Load data from the `winter_olympics_articles.json` file in the data folder.

In [7]:
# Load the data - articles from The Guardian
file_path = "data/"
file_name = "winter_olympics_articles.json"

with open(f"{file_path}{file_name}",'r', encoding='utf-8') as fp:
    articles = json.load(fp)

print(f"Loaded {len(articles)} articles from {file_name}")

Loaded 13 articles from winter_olympics_articles.json


Each dictionary entry includes the *title [date]* as `key` and the *body text* from the article as `value`.

In [8]:
article1 = list(articles.items())[0]
print("Key:",article1[0])
print("Value:",article1[1][:300],"...") # Just show first 300 characters

Key: Winter Olympics 2026: what you need to know if following from Australia [2026-02-02T14:00:10Z]
Value: When do the 2026 Winter Olympics start? The 2026 Winter Olympics officially open in the early hours of Saturday 7 February, Australian time, with the opening ceremony at Milan’s San Siro stadium. The Games run for two weeks, culminating in the closing ceremony on 23 February in Verona at the same ti ...


So the values gives us a list of documents that we can analyse.

In [9]:
# Get a list of documents
documents = list(articles.values())

# View first 400 characters of the 1st document
documents[0][:400]

'When do the 2026 Winter Olympics start? The 2026 Winter Olympics officially open in the early hours of Saturday 7 February, Australian time, with the opening ceremony at Milan’s San Siro stadium. The Games run for two weeks, culminating in the closing ceremony on 23 February in Verona at the same time of 6am AEDT. Several sports with packed schedules, including curling and ice hockey, begin a coup'

### Term Count 

**Finding important terms by the frequency of their occurance**

Using `CountVectorizer` create a `vector` for each document where the dimensionality of the vector is the `vocabulary` (all terms in the collection), and the value of each component is the number of times that the `term` occurs in the document.

All of these analyses, approach the document as a [Bag of Words](https://en.wikipedia.org/wiki/Bag-of-words_model) model. In this approach, the order of the words don't matter. A popular approach that takes into account order is [Word embedding](https://en.wikipedia.org/wiki/Word_embedding). This session does not explore word embedding.

In [10]:
# Only count terms that in maximum of 80% of documents, and a minimum of 2 documents. 
# Count a maximum of 10000 terms, and remove common english stop words
count_vectorizer = CountVectorizer(max_df=0.8, min_df=2, max_features=10000, stop_words="english")
count_dt_matrix = count_vectorizer.fit_transform(articles.values())

It is always important to understand the data we are working with

In [11]:
# Take a look at the vector for the first document
doc001_vector = count_dt_matrix.toarray()[0]
doc001_vector

array([1, 1, 0, 3, 2, 1, 0, 0, 0, 0, 0, 1, 0, 3, 0, 0, 6, 0, 0, 2, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0,
       0, 0, 1, 2, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1,
       5, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 2, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 2, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 0, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 4, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 1, 2, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       3, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 0, 2, 0, 3, 0, 0, 0, 0, 0,

In [12]:
# Get the 10000 terms identified during the vectorization process
feature_names = count_vectorizer.get_feature_names_out()
feature_names

array(['000', '10', '10th', '12', '15', '16', '17', '1994', '1998', '20',
       '2002', '2006', '2010', '2022', '2024', '2025', '2026', '20th',
       '21', '23', '25', '26', '27', '28', '30', '31', '50', '53', 'able',
       'accepted', 'according', 'accused', 'achieve', 'acknowledged',
       'acl', 'action', 'actually', 'added', 'adelaide', 'advertiser',
       'advice', 'aedt', 'aerial', 'aerials', 'afternoon', 'age', 'ago',
       'ahead', 'ahmad', 'air', 'alisa', 'allegedly', 'alongside',
       'alpine', 'alps', 'america', 'american', 'amid', 'annual',
       'anthony', 'appeal', 'arms', 'arrived', 'asian', 'assessment',
       'athlete', 'athletes', 'attack', 'attempt', 'attention',
       'australians', 'awarded', 'away', 'backed', 'baff', 'base',
       'based', 'battling', 'bearer', 'beat', 'beating', 'began', 'begin',
       'beijing', 'ben', 'best', 'better', 'bid', 'big', 'biggest', 'bit',
       'blood', 'blow', 'body', 'bradbury', 'brain', 'breadth',
       'breakthrou

In [13]:
# Look at how the counts match up to the terms (for the 1st doc)
doc001_term_counts = list(zip(feature_names,doc001_vector))
doc001_term_counts

[('000', np.int64(1)),
 ('10', np.int64(1)),
 ('10th', np.int64(0)),
 ('12', np.int64(3)),
 ('15', np.int64(2)),
 ('16', np.int64(1)),
 ('17', np.int64(0)),
 ('1994', np.int64(0)),
 ('1998', np.int64(0)),
 ('20', np.int64(0)),
 ('2002', np.int64(0)),
 ('2006', np.int64(1)),
 ('2010', np.int64(0)),
 ('2022', np.int64(3)),
 ('2024', np.int64(0)),
 ('2025', np.int64(0)),
 ('2026', np.int64(6)),
 ('20th', np.int64(0)),
 ('21', np.int64(0)),
 ('23', np.int64(2)),
 ('25', np.int64(0)),
 ('26', np.int64(0)),
 ('27', np.int64(0)),
 ('28', np.int64(0)),
 ('30', np.int64(0)),
 ('31', np.int64(0)),
 ('50', np.int64(0)),
 ('53', np.int64(1)),
 ('able', np.int64(0)),
 ('accepted', np.int64(0)),
 ('according', np.int64(0)),
 ('accused', np.int64(0)),
 ('achieve', np.int64(0)),
 ('acknowledged', np.int64(0)),
 ('acl', np.int64(0)),
 ('action', np.int64(0)),
 ('actually', np.int64(0)),
 ('added', np.int64(0)),
 ('adelaide', np.int64(0)),
 ('advertiser', np.int64(0)),
 ('advice', np.int64(0)),
 ('aedt'

In [14]:
# Take a look at the vocabulary which shows the total counts for whole collection
count_vectorizer.vocabulary_

{'2026': np.int64(16),
 'start': np.int64(624),
 'open': np.int64(464),
 'early': np.int64(211),
 'hours': np.int64(321),
 'saturday': np.int64(562),
 'february': np.int64(242),
 'time': np.int64(665),
 'opening': np.int64(465),
 'ceremony': np.int64(119),
 'milan': np.int64(421),
 'run': np.int64(556),
 'weeks': np.int64(712),
 'closing': np.int64(138),
 '23': np.int64(19),
 'aedt': np.int64(41),
 'sports': np.int64(617),
 'including': np.int64(329),
 'ice': np.int64(323),
 'begin': np.int64(82),
 'couple': np.int64(167),
 'days': np.int64(182),
 'mixed': np.int64(429),
 'team': np.int64(652),
 'just': np.int64(355),
 'missed': np.int64(425),
 'qualifying': np.int64(517),
 'despite': np.int64(194),
 'ranked': np.int64(525),
 'december': np.int64(185),
 'won': np.int64(721),
 'italy': np.int64(340),
 'final': np.int64(248),
 'event': np.int64(227),
 'men': np.int64(419),
 'game': np.int64(275),
 '12': np.int64(3),
 'related': np.int64(539),
 'youngest': np.int64(727),
 'olympian': np.i

#### Display matrix in dataframe

Take the term count matrix and display in a dataframe to make visible the structure


In [15]:
# Create a new dataframe with the matrix - use titles for the index and terms for the columns
count_df = pd.DataFrame(count_dt_matrix.toarray(), columns=feature_names)
count_df

,000,10,10th,12,15,16,17,1994,1998,20,...,woman,women,won,woods,work,worst,years,yellow,youngest,zealand
0,1,1,0,3,2,1,0,0,0,0,...,0,1,1,0,0,0,1,0,1,1
1,3,0,0,0,0,1,0,1,0,0,...,0,1,3,0,1,0,0,1,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,2,0,1,0,1,2,0,0,0
3,0,0,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,0,0,1,1,0,1,0,0,0,1,...,0,1,1,0,0,0,0,0,0,0
5,0,0,0,1,1,0,0,1,1,0,...,1,0,0,0,0,0,3,0,0,0
6,0,0,0,0,2,2,0,0,0,0,...,0,1,1,0,0,0,2,0,1,1
7,0,0,0,1,0,2,0,1,1,0,...,0,0,3,2,0,0,2,0,1,0
8,0,0,0,0,1,2,1,0,0,3,...,3,1,1,0,0,0,1,0,0,0
9,0,0,0,0,1,0,0,0,0,0,...,0,0,0,11,1,0,0,0,0,0


Keep a list of the titles of the articles to make it easy to see the headline that relates to the topics. We can always go back to the original documents if we need to.

In [16]:
titles=list(articles.keys())
titles

['Winter Olympics 2026: what you need to know if following from Australia [2026-02-02T14:00:10Z]',
 'Australia’s upward trajectory slips off course as Winter Olympics medal search goes on | Jack Snape [2026-02-12T03:36:06Z]',
 '‘Even more special’: Jakara Anthony dusts off Winter Olympics heartbreak for historic triumph [2026-02-15T01:32:16Z]',
 'A part-time job and DJ gigs helped Lara Hamilton reach the Winter Olympics. Now she wants to put Australia on the map [2026-02-17T14:00:15Z]',
 'Valentino Guseli’s unexpected big air dream ends without a medal in all-or-nothing final: ‘I left it all out there’ [2026-02-08T00:27:13Z]',
 'From Bradbury to Bright: five of Australia’s best Winter Olympic moments | Martin Pegan [2026-02-03T14:00:08Z]',
 'Australia’s youngest Winter Olympian Indra Brown: ‘I just love the feeling of flying’ | Martin Pegan [2026-02-01T14:00:34Z]',
 'How did Australia – better known for its beaches than snow – become a consistent Winter Olympics performer? | Kieran Pen

By selecting a row from the dataframe and sorting the values (counts), we can identify the top 10 terms

In [17]:
# Sample 5 random article numbers
samples = random.sample(range(0,len(count_df)),5)
samples

[2, 9, 5, 0, 7]

In [18]:
# View the associated terms
for sample in samples:
    doc = count_df.iloc[sample]
    title = titles[sample]
    top_terms = dict(count_df.iloc[sample].sort_values(ascending=False).head(10))
    print(f"[{sample}] {title}")
    print("\t- Top terms:",top_terms)
    print()

[2] ‘Even more special’: Jakara Anthony dusts off Winter Olympics heartbreak for historic triumph [2026-02-15T01:32:16Z]
	- Top terms: {'moguls': np.int64(13), 'anthony': np.int64(11), 'dual': np.int64(6), 'single': np.int64(5), 'event': np.int64(5), 'just': np.int64(4), 'final': np.int64(4), 'time': np.int64(4), 'course': np.int64(4), 'lot': np.int64(3)}

[9] Ice in his veins: Australian skier Cooper Woods embraces pressure to realise Winter Olympic dream [2026-02-13T02:17:26Z]
	- Top terms: {'woods': np.int64(11), 'said': np.int64(7), 'just': np.int64(7), 'cooper': np.int64(5), 'lot': np.int64(5), 'final': np.int64(4), 'win': np.int64(4), 'time': np.int64(4), 'perisher': np.int64(3), 'people': np.int64(3)}

[5] From Bradbury to Bright: five of Australia’s best Winter Olympic moments | Martin Pegan [2026-02-03T14:00:08Z]
	- Top terms: {'bradbury': np.int64(6), 'event': np.int64(5), 'high': np.int64(4), 'bronze': np.int64(4), 'skiing': np.int64(4), 'way': np.int64(3), 'speed': np.int64

#### Create a top10 terms dataframe

Using the index from the documents, create a dataframe that can hold the top10 terms for each document. We also include columns for our other analysis (tfidf, lda, nmf)

In [19]:
# Create a dataframe to hold top terms for each analysis type
terms_df = pd.DataFrame(index=count_df.index,columns=['title','count','tfidf','lda','nmf'])
terms_df['title'] = titles
terms_df

,title,count,tfidf,lda,nmf
0,Winter Olympics 2026: what you need to know if...,NaN,NaN,NaN,NaN
1,Australia’s upward trajectory slips off course...,NaN,NaN,NaN,NaN
2,‘Even more special’: Jakara Anthony dusts off ...,NaN,NaN,NaN,NaN
3,A part-time job and DJ gigs helped Lara Hamilt...,NaN,NaN,NaN,NaN
4,Valentino Guseli’s unexpected big air dream en...,NaN,NaN,NaN,NaN
5,From Bradbury to Bright: five of Australia’s b...,NaN,NaN,NaN,NaN
6,Australia’s youngest Winter Olympian Indra Bro...,NaN,NaN,NaN,NaN
7,How did Australia – better known for its beach...,NaN,NaN,NaN,NaN
8,‘Put the blinders on’: how Jakara Anthony can ...,NaN,NaN,NaN,NaN
9,Ice in his veins: Australian skier Cooper Wood...,NaN,NaN,NaN,NaN


Populate the count column with data created by the count vectorizer.

In [20]:
#For each doc, get the 10 columns with the largest counts
for idx in terms_df.index:
    counts = dict(count_df.loc[idx].sort_values(ascending=False).head(10))
    #print(counts)
    terms_df.at[idx,'count'] = list(counts.keys()) # Just the list of terms

terms_df

,title,count,tfidf,lda,nmf
0,Winter Olympics 2026: what you need to know if...,"[2026, ice, new, athletes, aedt, day, ski, wat...",NaN,NaN,NaN
1,Australia’s upward trajectory slips off course...,"[team, anthony, final, competition, medals, tr...",NaN,NaN,NaN
2,‘Even more special’: Jakara Anthony dusts off ...,"[moguls, anthony, dual, single, event, just, f...",NaN,NaN,NaN
3,A part-time job and DJ gigs helped Lara Hamilt...,"[hamilton, ski, skimo, just, running, says, sk...",NaN,NaN,NaN
4,Valentino Guseli’s unexpected big air dream en...,"[final, guseli, said, time, score, air, best, ...",NaN,NaN,NaN
5,From Bradbury to Bright: five of Australia’s b...,"[bradbury, event, high, bronze, skiing, way, s...",NaN,NaN,NaN
6,Australia’s youngest Winter Olympian Indra Bro...,"[just, indra, brown, time, says, want, halfpip...",NaN,NaN,NaN
7,How did Australia – better known for its beach...,"[medals, moguls, sports, team, funding, olympi...",NaN,NaN,NaN
8,‘Put the blinders on’: how Jakara Anthony can ...,"[anthony, moguls, says, ski, win, field, seaso...",NaN,NaN,NaN
9,Ice in his veins: Australian skier Cooper Wood...,"[woods, said, just, cooper, lot, final, win, t...",NaN,NaN,NaN


We should take a look at how some of the terms match up with the titles

In [21]:
# Sample 5 random articles
samples = random.sample(range(0,len(terms_df)),5)

for sample in samples:
    doc = terms_df.iloc[sample]
    print(f"[{sample}] {doc['title']}")
    print("\t>> Counts:\t",doc['count'])
    print()

[12] Morning Mail: Taylor sets up Liberal spill, millionaires call for higher taxes, Australia’s skiing hope slips up [2026-02-11T19:40:12Z]
	>> Counts:	 ['morning', 'smith', 'shock', 'australians', 'today', 'able', 'sydney', 'set', 'action', 'week']

[6] Australia’s youngest Winter Olympian Indra Brown: ‘I just love the feeling of flying’ | Martin Pegan [2026-02-01T14:00:34Z]
	>> Counts:	 ['just', 'indra', 'brown', 'time', 'says', 'want', 'halfpipe', 've', 'family', 'ski']

[4] Valentino Guseli’s unexpected big air dream ends without a medal in all-or-nothing final: ‘I left it all out there’ [2026-02-08T00:27:13Z]
	>> Counts:	 ['final', 'guseli', 'said', 'time', 'score', 'air', 'best', 'big', 'run', 'round']

[5] From Bradbury to Bright: five of Australia’s best Winter Olympic moments | Martin Pegan [2026-02-03T14:00:08Z]
	>> Counts:	 ['bradbury', 'event', 'high', 'bronze', 'skiing', 'way', 'speed', 'snowboard', 'snow', 'silver']

[8] ‘Put the blinders on’: how Jakara Anthony can make

### Term Frequency / Inverse Document Frequency (TF/IDF)

**Finding terms that are very common in a document, but less common in the whole collection**

The [TF/IDF](https://en.wikipedia.org/wiki/Tf–idf) algorithm takes the term frequencies for a document and divides them by the frequencies of the terms in the whole collection.


In [22]:
# Only count terms that in maximum of 80% of documents, and a minimum of 2 documents. 
# Count a maximum of 10000 terms, and remove common english stop words
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.8, min_df=2, max_features=10000, stop_words="english"
)

In [23]:
# Get the document vectors
tfidf_dt_matrix = tfidf_vectorizer.fit_transform(articles.values())

# Display the vector for the first document
tfidf_dt_matrix.toarray()[0]

array([0.05782947, 0.05782947, 0.        , 0.13860385, 0.07100469,
       0.03854199, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.05782947, 0.        , 0.10650704, 0.        ,
       0.        , 0.19692714, 0.        , 0.        , 0.10256164,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.05782947, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.2313179 , 0.        , 0.        , 0.        ,
       0.        , 0.042051  , 0.09240257, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.04620128, 0.05782947,
       0.05782947, 0.        , 0.        , 0.        , 0.03282119,
       0.        , 0.        , 0.        , 0.05782947, 0.        ,
       0.03854199, 0.17751174, 0.        , 0.05782947, 0.        ,
       0.042051  , 0.        , 0.        , 0.        , 0.     

#### Display matrix in dataframe

In [24]:
tfidf_df = pd.DataFrame(tfidf_dt_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
tfidf_df

,000,10,10th,12,15,16,17,1994,1998,20,...,woman,women,won,woods,work,worst,years,yellow,youngest,zealand
0,0.057829,0.057829,0.000000,0.138604,0.071005,0.038542,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.035502,0.030423,0.000000,0.000000,0.000000,0.030423,0.000000,0.051281,0.057829
1,0.171316,0.000000,0.000000,0.000000,0.000000,0.038059,0.000000,0.050639,0.000000,0.000000,...,0.000000,0.035058,0.090126,0.000000,0.057105,0.000000,0.000000,0.057105,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.071324,0.000000,0.046409,0.000000,0.058089,0.061119,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.046877,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.024661,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.053532,0.042768,0.000000,0.035678,0.000000,0.000000,0.000000,0.047470,...,0.000000,0.032864,0.028162,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.000000,0.000000,0.000000,0.046800,0.035962,0.000000,0.000000,0.051945,0.058579,0.000000,...,0.058579,0.000000,0.000000,0.000000,0.000000,0.000000,0.092451,0.000000,0.000000,0.000000
6,0.000000,0.000000,0.000000,0.000000,0.058372,0.063369,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.029186,0.025010,0.000000,0.000000,0.000000,0.050020,0.000000,0.042157,0.047541
7,0.000000,0.000000,0.000000,0.043702,0.000000,0.072914,0.000000,0.048507,0.054701,0.000000,...,0.000000,0.000000,0.086331,0.087404,0.000000,0.000000,0.057554,0.000000,0.048507,0.000000
8,0.000000,0.000000,0.000000,0.000000,0.033186,0.072055,0.054057,0.000000,0.000000,0.143806,...,0.162171,0.033186,0.028438,0.000000,0.000000,0.000000,0.028438,0.000000,0.000000,0.000000
9,0.000000,0.000000,0.000000,0.000000,0.032916,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.471197,0.053617,0.000000,0.000000,0.000000,0.000000,0.000000


#### Update the terms matrix

In [25]:
for idx in terms_df.index:
    tfidf = dict(tfidf_df.loc[idx].sort_values(ascending=False).head(10))
    #print(counts)
    terms_df.at[idx,'tfidf'] = list(tfidf.keys()) 

terms_df

,title,count,tfidf,lda,nmf
0,Winter Olympics 2026: what you need to know if...,"[2026, ice, new, athletes, aedt, day, ski, wat...","[ice, aedt, watch, 2026, athletes, table, moun...",NaN,NaN
1,Australia’s upward trajectory slips off course...,"[team, anthony, final, competition, medals, tr...","[anthony, team, competition, air, snowboarder,...",NaN,NaN
2,‘Even more special’: Jakara Anthony dusts off ...,"[moguls, anthony, dual, single, event, just, f...","[moguls, anthony, dual, single, course, event,...",NaN,NaN
3,A part-time job and DJ gigs helped Lara Hamilt...,"[hamilton, ski, skimo, just, running, says, sk...","[hamilton, skimo, running, ski, mountaineering...",NaN,NaN
4,Valentino Guseli’s unexpected big air dream en...,"[final, guseli, said, time, score, air, best, ...","[guseli, final, air, 50, landing, score, said,...",NaN,NaN
5,From Bradbury to Bright: five of Australia’s b...,"[bradbury, event, high, bronze, skiing, way, s...","[bradbury, smith, high, bronze, city, event, s...",NaN,NaN
6,Australia’s youngest Winter Olympian Indra Bro...,"[just, indra, brown, time, says, want, halfpip...","[indra, want, just, brown, says, ve, time, fam...",NaN,NaN
7,How did Australia – better known for its beach...,"[medals, moguls, sports, team, funding, olympi...","[medals, moguls, funding, aerial, sports, olym...",NaN,NaN
8,‘Put the blinders on’: how Jakara Anthony can ...,"[anthony, moguls, says, ski, win, field, seaso...","[anthony, says, moguls, field, makes, woman, s...",NaN,NaN
9,Ice in his veins: Australian skier Cooper Wood...,"[woods, said, just, cooper, lot, final, win, t...","[woods, cooper, said, just, lot, perisher, nsw...",NaN,NaN


#### Compare approaches

In [26]:
# Sample 5 random articles
samples = random.sample(range(0,len(terms_df)),5)

for sample in samples:
    doc = terms_df.iloc[sample]
    print(f"[{sample}] {doc['title']}")
    print("\t>> Counts:\t",doc['count'])
    print("\t>> TFIDF:\t",doc['tfidf'])
    print()

[11] Morning Mail: Lake Cargelligo suspect’s past revealed, supermarket ‘per unit’ prices under scrutiny, shark attack spike [2026-02-18T20:23:22Z]
	>> Counts:	 ['morning', 'history', 'including', 'new', 'government', 'people', 'day', 'children', 'crossword', 'asian']
	>> TFIDF:	 ['morning', 'children', 'government', 'history', 'people', 'update', 'reports', 'allegedly', 'uk', 'asian']

[5] From Bradbury to Bright: five of Australia’s best Winter Olympic moments | Martin Pegan [2026-02-03T14:00:08Z]
	>> Counts:	 ['bradbury', 'event', 'high', 'bronze', 'skiing', 'way', 'speed', 'snowboard', 'snow', 'silver']
	>> TFIDF:	 ['bradbury', 'smith', 'high', 'bronze', 'city', 'event', 'speed', 'nation', 'national', 'later']

[12] Morning Mail: Taylor sets up Liberal spill, millionaires call for higher taxes, Australia’s skiing hope slips up [2026-02-11T19:40:12Z]
	>> Counts:	 ['morning', 'smith', 'shock', 'australians', 'today', 'able', 'sydney', 'set', 'action', 'week']
	>> TFIDF:	 ['morning', 